<h1> Lab 1 Ex. 2: Satellite Image Super-Resolution </h1>

Unzip the dataset

**[TODO]** Import all the python modules you need

In [22]:
import os
import random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**[TODO]** Create a torch dataset. It must return a tuple with a float32 tensor for the low-resolution image patch and another float32 tensor for the high-resolution image patch. Patches values should be normalized (images are provided in the PNG file format and have intensity values in the [0,255] range).

Patch must be extracted as a 64x64 crop of the low-resolution image file at a random position. Note that the HR patch will be at 4x the resolution and its content should match the LR one.

Hint: do not use torchvision, but manually define the crop area by randomly generating the top-left coordinate so that you can match the LR and HR locations.

In [23]:
class SuperResolutionDataset(Dataset):
    def __init__(self, lr_dir, hr_dir):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir

        self.lr_images = sorted(os.listdir(lr_dir))
        self.hr_images = sorted(os.listdir(hr_dir))

        assert len(self.lr_images) == len(self.hr_images)

    def __len__(self):
        return len(self.lr_images)

    def __getitem__(self, idx):
        # Load images
        lr_path = os.path.join(self.lr_dir, self.lr_images[idx])
        hr_path = os.path.join(self.hr_dir, self.hr_images[idx])

        lr_img = np.array(Image.open(lr_path))
        hr_img = np.array(Image.open(hr_path))

        # Calculate the dimensions for cropping
        h, w, _ = lr_img.shape
        patch_size_lr = 64
        scale = 4
        patch_size_hr = patch_size_lr * scale

        # Randomly select top-left corner for cropping
        x = random.randint(0, w - patch_size_lr)
        y = random.randint(0, h - patch_size_lr)

        # Crop LR
        lr_patch = lr_img[y:y+patch_size_lr, x:x+patch_size_lr, :]

        # Corresponding HR coordinates
        x_hr = x * scale
        y_hr = y * scale

        # Crop HR
        hr_patch = hr_img[y_hr:y_hr+patch_size_hr, x_hr:x_hr+patch_size_hr, :]

        # Normalize to [0,1]
        lr_patch = lr_patch.astype(np.float32) / 255.0
        hr_patch = hr_patch.astype(np.float32) / 255.0

        # Convert to tensor
        lr_patch = torch.from_numpy(lr_patch).permute(2, 0, 1)
        hr_patch = torch.from_numpy(hr_patch).permute(2, 0, 1)

        return lr_patch, hr_patch

**[TODO]** Instantiate a Dataset and create a Dataloader for the training and test sets


In [24]:
train_dataset = SuperResolutionDataset(
    lr_dir="/content/drive/MyDrive/Formazione uni-lavorativa/Ingegneria informatica/ICT for smart/Primo anno secondo semestre/Lab advanced/train_sr/lr",
    hr_dir="/content/drive/MyDrive/Formazione uni-lavorativa/Ingegneria informatica/ICT for smart/Primo anno secondo semestre/Lab advanced/train_sr/hr"
)

test_dataset = SuperResolutionDataset(
    lr_dir="/content/drive/MyDrive/Formazione uni-lavorativa/Ingegneria informatica/ICT for smart/Primo anno secondo semestre/Lab advanced/test_sr/lr",
    hr_dir="/content/drive/MyDrive/Formazione uni-lavorativa/Ingegneria informatica/ICT for smart/Primo anno secondo semestre/Lab advanced/test_sr/hr"
)

train_loader =  torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

test_loader =  torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

**[TODO]** Define the neural network model. Implement the CNN with channel attention blocks describe in the lab instructions. Hints: you can use torch.nn.AdaptiveAvgPool2d(1) for the global average pool; you can create many torch.nn.Modules with pieces of the network that you then instantiate in the main network nn.Module.

In [10]:
import torch.nn as nn

#The channel attention mechanism
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.pool(x)
        w = self.mlp(w)
        return x * w

# The Residual Channel Attention Block (RCAB)
class RCAB(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1)
        )
        self.ca = ChannelAttention(channels)

    def forward(self, x):
        res = self.body(x)
        res = self.ca(res)
        return x + res


class ResidualGroup(nn.Module):
    def __init__(self, channels, num_rcab=3):
        super().__init__()

        self.rcabs = nn.Sequential(
            *[RCAB(channels) for _ in range(num_rcab)]
        )

        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        res = self.rcabs(x)
        res = self.conv(res)
        return x + res  # skip locale


class SRModel(nn.Module):
    def __init__(self, channels=64, scale=4):
        super().__init__()

        # Initial conv
        self.head = nn.Conv2d(3, channels, 3, padding=1)

        # Two residual groups
        self.group1 = ResidualGroup(channels)
        self.group2 = ResidualGroup(channels)

        # After groups
        self.conv_mid = nn.Conv2d(channels, channels, 3, padding=1)

        # Upsampling
        self.upscale = nn.Sequential(
            nn.Conv2d(channels, channels * (scale ** 2), 3, padding=1),
            nn.PixelShuffle(scale)
        )

        # Final conv
        self.tail = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, x):

        skip = nn.functional.interpolate(
            x, scale_factor=4, mode='bicubic', align_corners=False
        )

        # Head
        x = self.head(x)

        # Groups
        x = self.group1(x)
        x = self.group2(x)

        # Mid conv
        x = self.conv_mid(x)

        # Upscale
        x = self.upscale(x)
        x = self.tail(x)

        # Add global skip
        x = x + skip

        return x



**[TODO]** Instantiate the neural network model, move it GPU, and define an optimizer

In [25]:
model = SRModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sposto il modello sul device corretto
model = model.to(device)

criterion = torch.nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

**[TODO]** Train the model with an L1 loss

In [ ]:
# Test input
x = torch.randn(1, 3, 64, 64).to(device)
y = model(x)

print(y.shape)  # Let's see if the result is [1, 3, 256, 256]

#training
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    # we extract cropped patches from the original images
    for lr, hr in train_loader:
        lr = lr.to(device)
        hr = hr.to(device)

        #we apply them to the model
        sr = model(lr)

        #to limit the values in [0,1] for stability
        sr = torch.clamp(sr, 0.0, 1.0)

        # Loss
        loss = criterion(sr, hr)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}")



**[TODO]** Apply the SR network to the test images and compute the average PSNR of the SR images with respect to the HR ground truth. Visualize an LR image after 4x bicubic interpolation, the SR image and the HR ground truth.

In [ ]:

import torch.nn.functional as F
import math
import matplotlib.pyplot as plt



def to_img(tensor):
    img = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    return img


def compute_psnr(sr, hr):
    mse = F.mse_loss(sr, hr)
    if mse == 0:
        return 100
    return 10 * math.log10(1.0 / mse.item())


model.eval()

psnr_total = 0
num_images = 0

with torch.no_grad():
    for i, (lr, hr) in enumerate(test_loader):

        lr = lr.to(device)
        hr = hr.to(device)

        # Output della rete
        sr = model(lr)
        sr = torch.clamp(sr, 0.0, 1.0)

        # Calcolo PSNR
        psnr = compute_psnr(sr, hr)
        psnr_total += psnr
        num_images += 1

        if i < 2:
            print(f"Image {i+1} - PSNR: {psnr:.2f} dB")
            lr_up = F.interpolate(lr, scale_factor=4, mode='bicubic', align_corners=False)
            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.title("Bicubic x4")
            plt.imshow(to_img(lr_up))
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.title("Super-Resolution")
            plt.imshow(to_img(sr))
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.title("Ground Truth")
            plt.imshow(to_img(hr))
            plt.axis("off")

            plt.show()


avg_psnr = psnr_total / num_images
print("Average PSNR:", avg_psnr)



